# Hindi Text Restoration & Translation — Executed Demo (Interactive)

This notebook is an **executed evidence + interactive demo** for the project, based on a full run in **Google Colab** (artifacts synced into this repository).

- **Evidence sections** show saved outputs from generated artifacts in `data/`, `outputs/`, and `reports/`.
- **Interactive sections** let you run parts of *this repo’s* pipeline/code (if models/dependencies are available in your environment).

> **No off-the-shelf generative chat LLMs** (no GPT/Gemini/Claude). Uses ONNX punctuation model, mBERT pairwise+Borda ordering, and MarianMT/IndicTrans2 translation backends.  
> **Constraints:** CPU by default; if GPU is used, keep VRAM **< 1.5 GB** and measure it. Literal `NULL` markers (67 total) must be preserved.

## What the project does
1. Restore punctuation and sentence boundaries
2. Recover a candidate sentence order
3. Group every 5 sentences and translate to English
4. Produce aligned Hindi/English outputs

## 0. Setup (project root and imports)

In [1]:
from pathlib import Path
import os, sys, json

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('Project root:', ROOT)
print('Python:', sys.version.split()[0])
print('Exists src/:', (ROOT/'src').exists())

Project root: D:\Punctuation_Adder-and-Sentence-Ordering
Python: 3.x
Exists src/: True


## 1. Evidence: punctuation restoration (artifacts)
Raw vs punctuated preview from `data/raw/samples` and `data/punctuated`.

In [1]:
from pathlib import Path

def show(path,n=3):
    ls=Path(path).read_text(encoding='utf-8').splitlines()
    for i,l in enumerate(ls[:n],1): print(f'{i}: {l[:220]}')
    if len(ls)>n: print(f'... ({len(ls)-n} more lines)')

print('RAW sample-1 (head):')
show('data/raw/samples/sample-1.txt')
print('\nPUNCTUATED sample-1 (head):')
show('data/punctuated/sample-1.txt')

RAW sample-1 (head):
1: भगवान राम के पूर्वज महाराज शिवि ने अपना वानप्रस्थ गढ़ मुक्तेश्वर में ही व्यतीत किया था चेंगलपट स्टेशन से अरकोनम जाने वाली लाइन पर कांजीवरम स्टेशन है द्वारका चार धामों में एक धाम भी है कहते हैं कि उज्जैन में महाकाल को नमस
... (-2 more lines)

PUNCTUATED sample-1 (head):
1: भगवान राम के पूर्वज महाराज शिवि ने अपना वानप्रस्थ गढ़ मुक्तेश्वर में ही व्यतीत किया था। चेंगलपट स्टेशन से अरकोनम जाने वाली लाइन पर कांजीवरम स्टेशन है। द्वारका चार धामों में एक धाम भी है। कहते हैं कि उज्जैन में महाकाल को 
... (-2 more lines)


### 1b. Interactive: try punctuation restoration
Runs `src.punctuation.restore.PunctuationRestorer` on sample text (requires ONNX model files/deps).

In [ ]:
# Interactive: punctuation restoration
try:
    from src.punctuation.restore import PunctuationRestorer
    text = 'भारत एक विशाल देश है यह अनेक भाषाओं और संस्कृतियों का देश है'
    restorer = PunctuationRestorer()
    restored = restorer.restore(text)
    print('INPUT:   ', text)
    print('RESTORED:', restored)
except Exception as e:
    print('Interactive punctuation demo unavailable:', e)
    print('Tip: run where models/punctuation and dependencies are available.')

## 2. Evidence: ordering (mBERT pairwise + Borda)
Candidate ordered outputs and VRAM/resource evidence from the Colab run.

In [1]:
import json
from pathlib import Path
osum=json.load(open('data/ordered/ordering_summary.json',encoding='utf-8'))
keys=['model','device','batch_size','max_length','max_vram_budget_gb','any_over_budget','cuda_peak_allocated_gb_overall','cuda_peak_reserved_gb_overall']
print(json.dumps({k:osum.get(k) for k in keys if k in osum}, indent=2, ensure_ascii=False))
print('\nFirst 3 ordered sentences (sample-1):')
for i,l in enumerate(Path('data/ordered/sample-1.ordered.txt').read_text(encoding='utf-8').splitlines()[:3],1):
    print(f'{i}. {l[:220]}')

{
  "model": "bert-base-multilingual-cased",
  "device": "cuda",
  "batch_size": 16,
  "max_length": 256,
  "max_vram_budget_gb": 1.5,
  "any_over_budget": false,
  "cuda_peak_allocated_gb_overall": 0.741477,
  "cuda_peak_reserved_gb_overall": 1.050781
}

First 3 ordered sentences (sample-1):
1. मध्य रेलवे की मुंबई भोपाल दिल्ली लाइन के भोपाल स्टेशन से एक लाइन उज्जैन जाती है।
2. पश्चिम रेलवे की मुंबई कोटा दिल्ली लाइन पर नागदा स्टेशन से एक बड़ी लाइन भी उज्जैन तक गई है।
3. ये सातों धर्मस्थल सात नगरों या सप्तपुरियों के रूप में ग्रंथों में वर्णित हैं।


In [1]:
import json
osum=json.load(open('data/ordered/ordering_summary.json',encoding='utf-8'))
for f in osum.get('files',[]):
    print(f.get('input'), (f.get('ordering') or {}).get('cuda_memory'))
print('overall:', {k:osum.get(k) for k in ['cuda_peak_allocated_gb_overall','cuda_peak_reserved_gb_overall','any_over_budget']})

data/punctuated/sample-1.txt: {'label': 'sample-1.txt', 'peak_allocated_gb': 0.753472, 'peak_reserved_gb': 0.919922, 'budget_gb': 1.5, 'over_budget': False}
data/punctuated/sample-2.txt: {'label': 'sample-2.txt', 'peak_allocated_gb': 0.76456, 'peak_reserved_gb': 0.982422, 'budget_gb': 1.5, 'over_budget': False}
data/punctuated/sample-3.txt: {'label': 'sample-3.txt', 'peak_allocated_gb': 0.766639, 'peak_reserved_gb': 1.048828, 'budget_gb': 1.5, 'over_budget': False}
data/punctuated/sample-4.txt: {'label': 'sample-4.txt', 'peak_allocated_gb': 0.758996, 'peak_reserved_gb': 1.048828, 'budget_gb': 1.5, 'over_budget': False}
data/punctuated/sample-5.txt: {'label': 'sample-5.txt', 'peak_allocated_gb': 0.761976, 'peak_reserved_gb': 1.050781, 'budget_gb': 1.5, 'over_budget': False}
data/punctuated/sample-6.txt: {'label': 'sample-6.txt', 'peak_allocated_gb': 0.748141, 'peak_reserved_gb': 1.050781, 'budget_gb': 1.5, 'over_budget': False}
data/punctuated/sample-7.txt: {'label': 'sample-7.txt', 'pe

### 2b. Interactive: try ordering on a few sentences
Demonstrates `MBertPairwiseOrderer` on a small example (can be slow/heavy; downloads/loads mBERT if not cached).

In [ ]:
# Interactive: small ordering demo
try:
    import torch
    from src.ordering.order import MBertPairwiseOrderer, BERT_MODEL_NAME
    sentences = ['यह एक परीक्षण वाक्य है।', 'भारत एक विशाल देश है।', 'नई दिल्ली भारत की राजधानी है।']
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    orderer = MBertPairwiseOrderer(BERT_MODEL_NAME, device=device)
    order_ids, topics, meta = orderer.order(sentences)
    print('Device:', device)
    print('Input:   ', sentences)
    print('Ordered: ', [sentences[i] for i in order_ids])
    print('Indices:', order_ids, 'Meta:', meta)
except Exception as e:
    print('Interactive ordering demo unavailable/failed:', e)

## 3. Evidence: grouping and translation
Aligned outputs and translation backend details from the synced run.

In [1]:
import json
print(json.dumps({k: json.load(open('data/final/translation_summary.json',encoding='utf-8')).get(k) for k in ['translation_backend','translation_model','group_size']}, indent=2, ensure_ascii=False))
print('\nPer-sample (sentences/groups):')
for s in json.load(open('reports/run_summary.json',encoding='utf-8')).get('samples',[]):
    print(f"  sample {s['sample']}: sentences={s['sentences']} groups={s['groups']}")

{
  "translation_backend": "marian",
  "translation_model": "Helsinki-NLP/opus-mt-hi-en",
  "group_size": 5
}

Per-sample (sentences/groups):
  sample sample-1: sentences=50 groups=10
  sample sample-2: sentences=49 groups=10
  sample sample-3: sentences=47 groups=10
  sample sample-4: sentences=50 groups=10
  sample sample-5: sentences=52 groups=11
  sample sample-6: sentences=48 groups=10
  sample sample-7: sentences=60 groups=12


In [1]:
from pathlib import Path
for name in ['outputs/sample-1.sentences_hi.txt','outputs/sample-1.sentences_en.txt']:
    p=Path(name)
    print(name)
    print(p.read_text(encoding='utf-8').splitlines()[0][:300] if p.exists() else '[missing]')
    print()

outputs/sample-1.sentences_hi.txt
गढ़मुक्तेश्वर हिंदुओं का पावन तीर्थ है।

outputs/sample-1.sentences_en.txt
The sanctity of the gods of the gods of Mesopotamia is the Holy House of Freeman.


### 3b. Interactive: try translation
Attempts to load the configured translator (`src.translation.translate.load_translator`). Backend depends on environment/config (IndicTrans2 preferred if available; MarianMT fallback).

In [ ]:
# Interactive: translate a few Hindi sentences
try:
    from src.translation.translate import load_translator
    translator = load_translator()
    texts = ['भारत एक विशाल देश है।', 'यह एक परीक्षण वाक्य है।']
    out = translator.translate_texts(texts)
    print('Backend:', getattr(translator, 'backend', 'unknown'))
    print('Model:', getattr(translator, 'model_name', 'unknown'))
    for hi,en in zip(texts,out):
        print('HI:', hi)
        print('EN:', en)
        print()
except Exception as e:
    print('Interactive translation demo unavailable:', e)
    print('Tip: for IndicTrans2 use Colab with HF access/token and IndicTransToolkit installed.')

## 4. Evidence: aligned outputs and counts

In [1]:
from pathlib import Path
for i in range(1,8):
    hi=Path(f'outputs/sample-{i}.sentences_hi.txt').read_text(encoding='utf-8').splitlines()
    en=Path(f'outputs/sample-{i}.sentences_en.txt').read_text(encoding='utf-8').splitlines()
    gh=Path(f'outputs/sample-{i}.groups_hi.txt').read_text(encoding='utf-8').splitlines()
    ge=Path(f'outputs/sample-{i}.groups_en.txt').read_text(encoding='utf-8').splitlines()
    print(f'sample-{i}: sentences_hi={len(hi)} sentences_en={len(en)} groups_hi={len(gh)} groups_en={len(ge)} aligned={len(hi)==len(en) and len(gh)==len(ge)}')

sample-1: sentences_hi=50 sentences_en=50 groups_hi=10 groups_en=10 aligned=True
sample-2: sentences_hi=49 sentences_en=49 groups_hi=10 groups_en=10 aligned=True
sample-3: sentences_hi=47 sentences_en=47 groups_hi=10 groups_en=10 aligned=True
sample-4: sentences_hi=50 sentences_en=50 groups_hi=10 groups_en=10 aligned=True
sample-5: sentences_hi=52 sentences_en=52 groups_hi=11 groups_en=11 aligned=True
sample-6: sentences_hi=48 sentences_en=48 groups_hi=10 groups_en=10 aligned=True
sample-7: sentences_hi=60 sentences_en=60 groups_hi=12 groups_en=12 aligned=True


## 5. Evidence: NULL integrity and evaluation summary

In [1]:
import json
print(json.load(open('reports/evaluation.json',encoding='utf-8')).get('integrity',{}).get('stdout',''))

sample,raw_NULL,model_input_NULL,punctuated_NULL,ordered_NULL,final_hi_NULL,raw_chars,punct_chars,ordered_chars,ordered_sentences,groups
sample-1,5,5,5,5,5,3797,3851,3852,50,10
sample-2,5,5,5,5,5,3825,3881,3882,49,10
sample-3,6,6,6,6,6,3931,3989,3990,47,10
sample-4,4,4,4,4,4,5428,5504,5505,50,10
sample-5,9,9,9,9,9,4992,5079,5080,52,11
sample-6,11,11,11,11,11,2330,2422,2423,48,10
sample-7,27,27,27,27,27,3913,4000,4001,60,12

Note: This is a quick integrity check. Always inspect outputs manually.


In [1]:
import json
ev=json.load(open('reports/evaluation.json',encoding='utf-8'))
print('Punctuation:', json.dumps(ev.get('punctuation',{}), ensure_ascii=False))
print('Ordering:', json.dumps({k: ev.get('ordering',{}).get(k) for k in ['macro_avg_kendall_tau','macro_avg_pairwise_accuracy','macro_avg_matched_coverage','caveat']}, ensure_ascii=False))
print('Translation:', json.dumps(ev.get('translation',{}), ensure_ascii=False))

Punctuation: {"model": "Horizon-Labs/punctuation-restoration-small (quantized ONNX, CPU)", "proxy_evaluation": {"script": "scripts/evaluate_punctuation.py", "stdout": "", "stderr": "", "returncode": 0}, "note": "Proxy/approximate evaluation; not a gold-standard punctuation benchmark. Manual review required (especially NULL-adjacent and dialogue samples)."}
Ordering: {"macro_avg_kendall_tau": null, "macro_avg_pairwise_accuracy": null, "macro_avg_matched_coverage": null, "caveat": "Metrics are computed only on matched sentences. If references are not ground-truth original order, interpret as reference-alignment rather than true ordering accuracy."}
Translation: {"current_backend_generated_outputs": "Helsinki-NLP/opus-mt-hi-en (MarianMT fallback) for existing outputs in this environment", "implemented_preferred_backend": "IndicTrans2 (ai4bharat/indictrans2-indic-en-dist-200M, configurable) with fallback", "bleu": null, "chrf": null, "note": "No reference translations available in-repo; au

## 6. Where to get reference translations (for BLEU/chrF)?
This repo does **not** include gold/reference English translations for the provided samples. Without references aligned to the outputs you evaluate, automatic **BLEU/chrF cannot be computed** meaningfully.

**Options:**
- **Human references:** have a fluent Hindi–English translator create reference translations for the same (ordered) sentences/groups you score, store them consistently, then compute BLEU/chrF against those references.
- **Aligned parallel sources:** only if you can obtain references for the *same* source content/order (general corpora are not drop-in references for these specific assignment samples).
- **Manual/qualitative review:** use `reports/translation_review_template.csv` and spot checks for adequacy/fluency when references are unavailable.

## 7. How to regenerate with IndicTrans2 (Colab recommended)
1. Open `notebooks/colab_end_to_end.ipynb` (branch `main`) in Colab and run setup (`scripts/colab_setup.sh`).
2. On Hugging Face, **accept the license/access** for your IndicTrans2 model (default: `ai4bharat/indictrans2-indic-en-dist-200M`).
3. Add `HF_TOKEN` in Colab Secrets and set:
   ```python
   import os
   os.environ['TRANSLATION_BACKEND'] = 'indictrans2'  # or 'auto'
   os.environ['INDICTRANS2_MODEL_ID'] = 'ai4bharat/indictrans2-indic-en-dist-200M'
   ```
4. Run stages (punctuation → ordering → grouping+translation). If using GPU, keep VRAM **< 1.5 GB** (use `ORDERING_DEVICE`, `ORDERING_BATCH_SIZE`, `ORDERING_MAX_LENGTH`, `ORDERING_MAX_VRAM_GB` and check logs).
5. Run: `python src/group_and_translate.py` (prefers IndicTrans2 when available), then `python scripts/generate_aligned_outputs.py`, `python scripts/compute_metrics.py`, `python scripts/verify_integrity.py`.
6. Verify in `data/final/translation_summary.json` (and per-sample pipeline JSON) that `translation_backend` is `indictrans2` and the model ID is correct.
7. Sync/download updated `outputs/`, `reports/`, `data/ordered/`, `data/final/` (and notebook outputs) back to the repo. Rebuild/refresh demo (`python scripts/build_executed_demo.py` or update this notebook) and commit/push.

> If access/toolkit is missing, the pipeline may fall back to MarianMT—always confirm via logs and `translation_summary.json`.

## 8. Reproduce
- **Colab:** `notebooks/colab_end_to_end.ipynb`
- **Local:** `python -m src.pipeline` (or run stage scripts individually)
- **Checks:** `scripts/verify_integrity.py`, `scripts/compute_metrics.py`
- **Demo from artifacts:** `scripts/build_executed_demo.py`

**Notes:** keep `NULL` markers preserved (67), report backend/model/metrics honestly, and measure GPU VRAM if used (< 1.5 GB).